# PWV03d_longscale : Two-Point Temporal Correlation — long timescales (Sylvie's estimator)

## Overview

This notebook is the long-timescale counterpart of
`PWV03d_TwoPoint_TemporalCorrelation_separateFilters_sylvie.ipynb`.
It uses **Sylvie Dagoret-Campagne's own pair-based DCF estimator**
(`ComputePWVAndTimeDifference` + `ComputeMyDCF_PwvixPwvj`) to measure the
two-point temporal correlation of the PWV at Cerro Pachon on
**multi-day timescales** (from 1 minute to ~500 days).

## Key differences from PWV03d (intra-night)

In the intra-night version, per-night mean subtraction removes the
slow PWV modulation and only same-night pairs are used (lags ≤ 10 h).  Here:

1. **Global mean subtraction**: only the overall dataset mean is subtracted,
   so both intra-night fluctuations and the inter-night PWV modulation
   (synoptic weather, seasonal cycle) contribute to the variance and
   to the pair products.  This allows the correlation to be measured at
   all timescales simultaneously.

2. **All pairs are included** (no same-night restriction): `ComputePWVAndTimeDifference_vectorized`
   forms upper-triangle pairs over the entire sorted time array, so lags
   range from the minimum cadence to the full multi-year span.

3. **Long logarithmic lag grid**: 80 bins from 1 min to 500 days.

4. Both the raw correlation and a per-filter breakdown are provided.

## Estimator

$$
C(\Delta t_k) =
  \frac{1}{N_k\,\sigma^2}
  \sum_{\{i,j\}\in\mathrm{bin}_k}
    \delta\mathrm{PWV}_i\,\delta\mathrm{PWV}_j,
  \qquad
  \delta\mathrm{PWV} = \mathrm{PWV} - \langle\mathrm{PWV}\rangle_{\mathrm{global}}
$$

where $\sigma^2$ is the global variance (including inter-night variability).

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-29  
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

%matplotlib inline

mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
pathfigs = 'figs_PWV03longscale'
prefix   = 'pwv03d_longscale_sylvie'
os.makedirs(pathfigs, exist_ok=True)
figtype  = '.pdf'

In [ ]:
FLAG_WITHCOLLIMATOR     = True
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00.0+0000')

# Instrumental repeatability sigma (mm) — added in quadrature
SIGMA_repeat = 0.12

---
## 1. Load data & build Time column

In [ ]:
import sys
sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split('/')[-1]
print(f'Loading: {atmfilename}')

if 'parquet' in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

df_spec['Time'] = pd.to_datetime(df_spec['DATE-OBS'], utc=True)
df_spec = df_spec.sort_values('Time', ascending=True).reset_index(drop=True)

if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['Time'] > datetime_WITHCOLLIMATOR].copy()

print(f'Shape: {df_spec.shape}')
print(f'Date range: {df_spec["Time"].min().date()}  ->  {df_spec["Time"].max().date()}')

In [ ]:
df_spec['nightObs'] = df_spec.apply(lambda x: x['id'] // 100_000, axis=1)
df_spec['seq_num']  = df_spec['id'] % 100_000
df_spec['night_from_time_utc'] = df_spec['Time'].dt.strftime('%Y%m%d').astype(int)

FLAG_CORRECT_NIGHTOBS_VARIABLES = True
if FLAG_CORRECT_NIGHTOBS_VARIABLES and 'run2026_v02' in version_run:
    print('COMPUTE NIGHTOBS FROM Time')
    df_spec['Time_local'] = df_spec['Time'].dt.tz_convert('America/Santiago')
    df_spec['nightObs_corr'] = (
        (df_spec['Time_local'] - pd.Timedelta(hours=12))
        .dt.strftime('%Y%m%d')
        .astype(int)
    )
    df_spec['nightObs'] = df_spec['nightObs_corr']

In [ ]:
if 'run2026_v02' in version_run:
    df_spec['chi2_ram'] = df_spec['CHI2_FIT']
    df_spec['chi2_rum'] = df_spec['CHI2_FIT']

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec['FILTER'].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ['PWV [mm]_ram', 'PWV [mm]_rum', 'PWV [mm]_err_ram', 'PWV [mm]_err_rum']
df_spec.dropna(subset=pwv_cols, inplace=True)
print(f'After filter selection: {len(df_spec)} rows')

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='CHI2_FIT', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='chi2_ram', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='chi2_rum', ext='norm')

---
## 2. Quality cuts

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

pathdata = 'data_PWV01seas'
if FLAG_LOOSE_CUTS:
    filename_cuts = f'{pathdata}/cuts_loose_finaldecision.json'
    cutstype_name = 'loose-cuts'
elif FLAG_TIGHT_CUTS:
    filename_cuts = f'{pathdata}/cuts_tight_finaldecision.json'
    cutstype_name = 'tight-cuts'
else:
    filename_cuts = f'{pathdata}/cuts_finaldecision.json'
    cutstype_name = 'standard-cuts'

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col='id')
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on='id')
df_keep     = df_selected[df_selected['pass_all_cuts']].copy()

print(f'{cutstype_name}: {len(df_keep)} / {len(df_spec)} kept')

---
## 3. Prepare PWV time series — global mean subtraction

Unlike `PWV03d` (per-night subtraction), we subtract only the **global
mean** so that the inter-night PWV modulation is preserved in the
residuals $\delta\mathrm{PWV}$.  The normalisation variance $\sigma^2$
includes both intra-night and inter-night variability.

In [ ]:
df_keep['PWV']     = df_keep['PWV [mm]_ram']
df_keep['PWV_err'] = df_keep['PWV [mm]_err_ram']

# Time in minutes since first observation (consistent with PWV03d convention)
t0_min            = df_keep['MJD'].min() * 1440.0
df_keep['t_min']  = df_keep['MJD'] * 1440.0 - t0_min
df_keep['t_day']  = df_keep['t_min'] / 1440.0

# Global mean subtraction
global_mean       = df_keep['PWV'].mean()
df_keep['dPWV']   = df_keep['PWV'] - global_mean

# Total variance (intra + inter night)
pwv_var = df_keep['dPWV'].var(ddof=1) + SIGMA_repeat**2
sigma_pwv = np.sqrt(pwv_var)

n_nights = df_keep['nightObs'].nunique()
print(f'N observations              : {len(df_keep)}')
print(f'N nights                    : {n_nights}')
print(f'Global mean PWV             : {global_mean:.3f} mm')
print(f'Total sigma (intra+inter)   : {sigma_pwv:.3f} mm')
print(f'Time span                   : {df_keep["t_day"].max():.1f} days')

In [ ]:
# Quick sanity plot
fig0, axes0 = plt.subplots(1, 2, figsize=(16, 4))

axes0[0].scatter(df_keep['t_day'], df_keep['PWV'],
                 s=2, alpha=0.4, color='steelblue')
axes0[0].axhline(global_mean, color='darkred', lw=1.5, ls='--',
                 label=f'Global mean = {global_mean:.2f} mm')
axes0[0].set_xlabel('Time [days since first obs]', fontsize=12)
axes0[0].set_ylabel('PWV [mm]', fontsize=12)
axes0[0].set_title('PWV time series (global mean subtraction)', fontsize=11)
axes0[0].legend(fontsize=9)

axes0[1].hist(df_keep['dPWV'], bins=60, color='steelblue', alpha=0.8,
              edgecolor='white', linewidth=0.3)
axes0[1].axvline(0, color='black', ls='--', lw=1)
axes0[1].set_xlabel(r'$\delta$PWV [mm]', fontsize=12)
axes0[1].set_ylabel('Count', fontsize=12)
axes0[1].set_title(rf'$\delta$PWV residuals  $\sigma = {sigma_pwv:.3f}$ mm', fontsize=11)

fig0.suptitle(f'Global mean subtraction — {version_run}', fontsize=12, y=1.02)
fig0.tight_layout()
fig0.savefig(f'{pathfigs}/{prefix}_{version_run}_global_mean_subtraction.pdf',
             bbox_inches='tight')
plt.show()

---
## 4. Sylvie's DCF functions — vectorised all-pairs version

In [ ]:
def ComputePWVAndTimeDifference_vectorized_allpairs(df, t_col='t_min', dpwv_col='dPWV',
                                                     lag_min_min=1.0, lag_max_min=None,
                                                     n_max_sub=5000, rng=None):
    """
    Vectorised computation of all upper-triangle pairs (i < j) over the full dataset.
    No night-boundary restriction: cross-night pairs are included.

    Optionally sub-samples to n_max_sub observations to limit memory usage.

    Parameters
    ----------
    df           : pd.DataFrame  — must contain t_col and dpwv_col.
    t_col        : str           — time column (minutes).
    dpwv_col     : str           — global-mean-subtracted PWV column.
    lag_min_min  : float         — minimum lag to keep (minutes).
    lag_max_min  : float or None — maximum lag to keep (minutes, None = no cut).
    n_max_sub    : int           — maximum observations to use (sub-sample if larger).
    rng          : np.Generator  — random number generator for reproducibility.

    Returns
    -------
    dt_pairs, prod_pairs : np.ndarray — time lags (min) and products (mm^2).
    """
    if rng is None:
        rng = np.random.default_rng(seed=42)

    t_arr   = df[t_col].values
    dp_arr  = df[dpwv_col].values
    N       = len(t_arr)

    if N > n_max_sub:
        idx = np.sort(rng.choice(N, n_max_sub, replace=False))
        t_arr  = t_arr[idx]
        dp_arr = dp_arr[idx]
        print(f'  Sub-sampled to {n_max_sub} obs -> {n_max_sub*(n_max_sub-1)//2:,} pairs')

    ii, jj   = np.triu_indices(len(t_arr), k=1)
    dt_pairs = t_arr[jj]  - t_arr[ii]     # always >= 0
    pr_pairs = dp_arr[ii] * dp_arr[jj]

    ok = dt_pairs >= lag_min_min
    if lag_max_min is not None:
        ok &= (dt_pairs <= lag_max_min)

    return dt_pairs[ok], pr_pairs[ok]


def ComputeMyDCF_PwvixPwvj(list_of_bins, dt_pairs, prod_pairs, sigPWV):
    """
    Bin pair products and compute the normalised DCF.

    Ported and generalised from Sylvie's OLD notebook implementation.

    Parameters
    ----------
    list_of_bins : array-like  — bin edges (same unit as dt_pairs).
    dt_pairs     : np.ndarray  — time lags.
    prod_pairs   : np.ndarray  — products dPWV_i * dPWV_j.
    sigPWV       : float       — normalisation sigma (mm).

    Returns
    -------
    xcenter  : bin centres (arithmetic mean of edges).
    ydata    : normalised DCF C(Delta_t).
    ydataerr : standard error on C(Delta_t).
    ncount   : number of pairs per bin.
    """
    Nbins    = len(list_of_bins)
    xcenter  = (list_of_bins[:-1] + list_of_bins[1:]) / 2.0
    N        = len(xcenter)
    ydata    = np.full(N, np.nan)
    ydataerr = np.full(N, np.nan)
    ncount   = np.zeros(N, dtype=int)

    var = sigPWV**2

    for ibin in range(N):
        sel    = (dt_pairs >= list_of_bins[ibin]) & (dt_pairs < list_of_bins[ibin + 1])
        all_y  = prod_pairs[sel] / var
        n      = sel.sum()
        ncount[ibin]   = n
        ydata[ibin]    = all_y.mean() if n > 0 else np.nan
        ydataerr[ibin] = (all_y.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan

    return xcenter, ydata, ydataerr, ncount

---
## 5. Compute DCF — all-pairs, long lag grid

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
LAG_MIN_MIN  =    1.0            # 1 minute
LAG_MAX_DAY  =  180.0            # 500 days
LAG_MAX_MIN  = LAG_MAX_DAY * 1440.0
N_BINS_LOG   =   60             # log-spaced bins
MIN_PAIRS    =   10              # minimum pairs per bin for display
N_MAX_SUB    = 5000              # sub-sampling limit

rng = np.random.default_rng(seed=42)

# Bin edges (log-spaced in minutes)
bin_edges       = np.logspace(np.log10(LAG_MIN_MIN), np.log10(LAG_MAX_MIN), N_BINS_LOG + 1)
bin_centers_min = np.sqrt(bin_edges[:-1] * bin_edges[1:])   # geometric mean
bin_centers_day = bin_centers_min / 1440.0
bin_centers_hr  = bin_centers_min / 60.0

print(f'Lag range: {LAG_MIN_MIN:.0f} min – {LAG_MAX_DAY:.0f} days')
print(f'N log bins: {N_BINS_LOG}')
PLOT_XSCALE = "log"
PLOT_YMIN = -0.5
PLOT_YMAX = 1.2

In [ ]:
# ── Form all pairs ─────────────────────────────────────────────────────────────
print(f'Computing all pairs over {len(df_keep)} observations (with optional sub-sampling)...')
dt_all, pr_all = ComputePWVAndTimeDifference_vectorized_allpairs(
    df_keep, t_col='t_min', dpwv_col='dPWV',
    lag_min_min=LAG_MIN_MIN, lag_max_min=LAG_MAX_MIN,
    n_max_sub=N_MAX_SUB, rng=rng
)
print(f'Total pairs within [{LAG_MIN_MIN:.0f} min, {LAG_MAX_DAY:.0f} days]: {len(dt_all):,}')

In [ ]:
# ── Bin and normalise ──────────────────────────────────────────────────────────
xcenter, corr, corr_err, ncount = ComputeMyDCF_PwvixPwvj(
    bin_edges, dt_all, pr_all, sigPWV=sigma_pwv
)

good = ncount >= MIN_PAIRS
print(f'Bins with >= {MIN_PAIRS} pairs: {good.sum()} / {N_BINS_LOG}')

# 1/e crossing on the long-scale grid
inv_e     = 1.0 / np.e
cross_idx = np.where(good & (corr < inv_e))[0]
if len(cross_idx) > 0:
    tau_min_1e = bin_centers_min[cross_idx[0]]
    tau_day_1e = bin_centers_day[cross_idx[0]]
    print(f'First 1/e crossing: {tau_min_1e:.1f} min = {tau_day_1e:.3f} days')
else:
    tau_min_1e = np.nan
    tau_day_1e = np.nan
    print('C(Delta_t) does not drop below 1/e in the measured lag range')

---
## 6. Main figures — full range and zooms

In [ ]:
CORR_COLOR    = 'steelblue'
ZERO_COLOR    = 'gray'
INV_E_COLOR   = 'darkorange'
ONE_DAY_COLOR = 'purple'
WEEK_COLOR    = 'green'

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.subplots_adjust(wspace=0.32)

# ── LEFT: full range, log days ─────────────────────────────────────────────────
ax_l = axes[0]
ax_l.fill_between(
    bin_centers_day[good],
    corr[good] - corr_err[good],
    corr[good] + corr_err[good],
    alpha=0.25, color=CORR_COLOR
)
ax_l.plot(bin_centers_day[good], corr[good],
          color=CORR_COLOR, lw=1.8, marker='o', ms=3, zorder=3,
          label=r'$C(\Delta t)$ [Sylvie]')
ax_l.axhline(0.0,        ls='--', color=ZERO_COLOR,    lw=1.0)
ax_l.axhline(inv_e,      ls=':',  color=INV_E_COLOR,   lw=1.5, label=r'$C=1/e$')
ax_l.axvline(1.0,        ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.6, label='1 day')
ax_l.axvline(7.0,        ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.6, label='7 days')
ax_l.axvline(30.0,       ls='--', color=WEEK_COLOR,    lw=1.0, alpha=0.5, label='30 days')
ax_l.axvline(365.25,     ls='--', color='red',          lw=1.0, alpha=0.5, label='1 year')
if np.isfinite(tau_day_1e):
    ax_l.axvline(tau_day_1e, ls='-', color='darkred', lw=1.5,
                 label=rf'$\tau_{{1/e}} = {tau_day_1e:.2f}$ d')
ax_l.set_xscale(PLOT_XSCALE)
ax_l.set_xlim(LAG_MIN_MIN / 1440.0, LAG_MAX_DAY)
ax_l.set_ylim(PLOT_YMIN, PLOT_YMAX)
ax_l.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax_l.set_ylabel(r'$C(\Delta t)$', fontsize=13)
ax_l.set_title('Full range — log scale (days)', fontsize=11)
ax_l.legend(fontsize=9, loc='upper right')

# ── RIGHT: zoom on intra-night (1–600 min), log minutes ───────────────────────
ax_r = axes[1]
zoom_mask = good & (bin_centers_min <= 600.0)

ax_r.fill_between(
    bin_centers_min[zoom_mask],
    corr[zoom_mask] - corr_err[zoom_mask],
    corr[zoom_mask] + corr_err[zoom_mask],
    alpha=0.25, color=CORR_COLOR
)
ax_r.plot(bin_centers_min[zoom_mask], corr[zoom_mask],
          color=CORR_COLOR, lw=1.8, marker='o', ms=4, zorder=3,
          label=r'$C(\Delta t)$ [Sylvie]')
ax_r.axhline(0.0,  ls='--', color=ZERO_COLOR,  lw=1.0)
ax_r.axhline(inv_e, ls=':',  color=INV_E_COLOR, lw=1.5, label=r'$C=1/e$')
ax_r.set_xscale(PLOT_XSCALE)
ax_r.set_xlim(LAG_MIN_MIN, 600.)
ax_r.set_ylim(PLOT_YMIN, PLOT_YMAX)
ax_r.set_xlabel(r'$\Delta t$ [minutes]', fontsize=13)
ax_r.set_ylabel(r'$C(\Delta t)$', fontsize=13)
ax_r.set_title("Intra-night zoom (1–600 min)", fontsize=11)
ax_r.legend(fontsize=9, loc='upper right')

fig.suptitle(
    f'PWV long-scale DCF [Sylvie] — global mean subtracted, all pairs\n{version_run}',
    fontsize=12, y=1.02
)
figfile1 = f'{pathfigs}/{prefix}_{version_run}_dcf_longscale.pdf'
fig.savefig(figfile1, bbox_inches='tight')
print(f'Saved: {figfile1}')
plt.show()

In [ ]:
# Zoom: multi-day regime (0.5 – 100 days)
fig2, ax2 = plt.subplots(figsize=(12, 5))

day_mask = good & (bin_centers_day >= 0.4) & (bin_centers_day <= 100.0)

ax2.fill_between(
    bin_centers_day[day_mask],
    corr[day_mask] - corr_err[day_mask],
    corr[day_mask] + corr_err[day_mask],
    alpha=0.3, color=CORR_COLOR
)
ax2.plot(bin_centers_day[day_mask], corr[day_mask],
         color=CORR_COLOR, lw=2, marker='o', ms=5, zorder=3,
         label=r'$C(\Delta t)$ [Sylvie]')
ax2.axhline(0.0,  ls='--', color=ZERO_COLOR,  lw=1.0)
ax2.axhline(inv_e, ls=':',  color=INV_E_COLOR, lw=1.5, label=r'$C=1/e$')
for d, lbl, col in [(1.0, '1 day', ONE_DAY_COLOR),
                     (7.0, '7 days', WEEK_COLOR),
                     (30.0, '30 days', 'olive')]:
    ax2.axvline(d, ls='--', color=col, lw=1.2, alpha=0.7, label=lbl)

ax2.set_xscale(PLOT_XSCALE)
ax2.set_xlim(0.3, 100.0)
ax2.set_ylim(PLOT_YMIN, PLOT_YMAX)
ax2.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax2.set_ylabel(r'$C(\Delta t)$', fontsize=13)
ax2.set_title('Multi-day PWV correlation (0.4 – 100 days)', fontsize=11)
ax2.legend(fontsize=9, loc='upper right')
fig2.suptitle(f'PWV long-scale DCF [Sylvie] — {version_run}', fontsize=12)
figfile2 = f'{pathfigs}/{prefix}_{version_run}_dcf_multiday.pdf'
fig2.savefig(figfile2, bbox_inches='tight')
print(f'Saved: {figfile2}')
plt.show()

---
## 7. Pair count per bin

In [ ]:
fig3, ax3 = plt.subplots(figsize=(14, 4))
ax3.bar(np.arange(N_BINS_LOG)[good], ncount[good],
        width=0.8, color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.4)
ax3.set_yscale('log')
ax3.set_xlabel('Bin index', fontsize=12)
ax3.set_ylabel('Number of pairs', fontsize=12)
ax3.set_title(f'Pair count per log-lag bin ({LAG_MIN_MIN:.0f} min – {LAG_MAX_DAY:.0f} days)', fontsize=11)

tick_idx = np.linspace(0, N_BINS_LOG - 1, 10, dtype=int)
ax3.set_xticks(tick_idx)
ax3.set_xticklabels(
    [f'{bin_centers_min[k]:.0f} min' if bin_centers_day[k] < 1.0
     else f'{bin_centers_day[k]:.1f} d'
     for k in tick_idx],
    rotation=35, ha='right', fontsize=9
)
fig3.tight_layout()
figfile3 = f'{pathfigs}/{prefix}_{version_run}_pair_counts.pdf'
fig3.savefig(figfile3, bbox_inches='tight')
print(f'Saved: {figfile3}')
plt.show()

---
## 8. Per-filter DCF — long-scale

In [ ]:
FILTERS_OF_INTEREST = ['empty', 'OG550_65mm_1', 'FELH0600']
filter_color = {
    'empty'        : '#1f77b4',
    'OG550_65mm_1' : '#d62728',
    'FELH0600'     : '#2ca02c',
}
filter_label = {
    'empty'        : 'empty',
    'OG550_65mm_1' : 'OG550',
    'FELH0600'     : 'FELH0600',
}

corr_by_filter     = {}
corr_err_by_filter = {}
ncount_by_filter   = {}

for filt in FILTERS_OF_INTEREST:
    sub_f = df_keep[df_keep['FILTER'] == filt].copy()
    if len(sub_f) < 10:
        print(f'  {filt}: too few points, skipped')
        continue

    # Global mean subtraction per filter
    gm_f            = sub_f['PWV'].mean()
    sub_f['dPWV_f'] = sub_f['PWV'] - gm_f
    var_f           = sub_f['dPWV_f'].var(ddof=1) + SIGMA_repeat**2
    sig_f           = np.sqrt(var_f)

    print(f'  {filt}: {len(sub_f)} obs, sigma = {sig_f:.3f} mm')
    dt_f, pr_f = ComputePWVAndTimeDifference_vectorized_allpairs(
        sub_f, t_col='t_min', dpwv_col='dPWV_f',
        lag_min_min=LAG_MIN_MIN, lag_max_min=LAG_MAX_MIN,
        n_max_sub=N_MAX_SUB, rng=rng
    )
    _, c_f, ce_f, nc_f = ComputeMyDCF_PwvixPwvj(bin_edges, dt_f, pr_f, sigPWV=sig_f)

    corr_by_filter[filt]     = c_f
    corr_err_by_filter[filt] = ce_f
    ncount_by_filter[filt]   = nc_f
    print(f'    -> {(nc_f >= MIN_PAIRS).sum()} good bins')

print('Done.')

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(18, 6))
fig4.subplots_adjust(wspace=0.32)

for ax4, xlim4, title4 in [
    (axes4[0], (LAG_MIN_MIN / 1440.0, LAG_MAX_DAY), 'Full range — log days'),
    (axes4[1], (0.4, 100.0),                         'Zoom 0.4–100 days'),
]:
    ax4.plot(bin_centers_day[good], corr[good],
             color='black', lw=1.0, alpha=0.35, ls='--', label='All filters')
    for filt in FILTERS_OF_INTEREST:
        if filt not in corr_by_filter:
            continue
        c_f  = corr_by_filter[filt]
        ce_f = corr_err_by_filter[filt]
        gf   = ncount_by_filter[filt] >= MIN_PAIRS
        ax4.fill_between(bin_centers_day[gf],
                         c_f[gf] - ce_f[gf], c_f[gf] + ce_f[gf],
                         alpha=0.15, color=filter_color[filt])
        ax4.plot(bin_centers_day[gf], c_f[gf],
                 color=filter_color[filt], lw=1.8, marker='o', ms=3,
                 label=filter_label[filt])
    ax4.axhline(0.0,  ls='--', color=ZERO_COLOR,    lw=1.0)
    ax4.axhline(inv_e, ls=':',  color=INV_E_COLOR,   lw=1.5, label=r'$C=1/e$')
    ax4.axvline(1.0,  ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.5, label='1 day')
    ax4.axvline(7.0,  ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.5, label='7 days')
    ax4.set_xscale(PLOT_XSCALE)
    ax4.set_xlim(xlim4)
    ax4.set_ylim(PLOT_YMIN, PLOT_YMAX)
    ax4.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
    ax4.set_ylabel(r'$C(\Delta t)$', fontsize=13)
    ax4.set_title(title4, fontsize=11)
    ax4.legend(fontsize=9, loc='upper right')

fig4.suptitle(f'PWV long-scale DCF per filter [Sylvie] — {version_run}', fontsize=12, y=1.01)
figfile4 = f'{pathfigs}/{prefix}_{version_run}_dcf_per_filter.pdf'
fig4.savefig(figfile4, bbox_inches='tight')
print(f'Saved: {figfile4}')
plt.show()

---
## 9. Summary & export

In [ ]:
df_corr_out = pd.DataFrame({
    'lag_min' : bin_centers_min,
    'lag_hr'  : bin_centers_hr,
    'lag_day' : bin_centers_day,
    'C'       : corr,
    'C_err'   : corr_err,
    'n_pairs' : ncount,
})
csv_out = f'{pathfigs}/{prefix}_{version_run}_dcf_longscale.csv'
df_corr_out.to_csv(csv_out, index=False, float_format='%.6f')
print(f'Saved: {csv_out}')

print(f'\n=== PWV long-scale DCF [Sylvie] summary ===')
print(f'  N observations (after cuts) : {len(df_keep)}')
print(f'  N nights                    : {n_nights}')
print(f'  Global mean PWV             : {global_mean:.3f} mm')
print(f'  Total sigma                 : {sigma_pwv:.3f} mm')
print(f'  Total pairs after lag cut   : {len(dt_all):,}')
print(f'  Lag range                   : {LAG_MIN_MIN:.0f} min – {LAG_MAX_DAY:.0f} days')
print(f'  N log bins                  : {N_BINS_LOG}')
print(f'  Good bins (>= {MIN_PAIRS} pairs): {good.sum()} / {N_BINS_LOG}')
if np.isfinite(tau_day_1e):
    print(f'  1/e crossing (empirical)    : {tau_day_1e:.3f} days = {tau_min_1e:.0f} min')